# Model

## Model Requirements


In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [ ]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers>=4.33.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

Restart the IDE - Go to runtime -> Restart Session

Run the next cells

## Model Initialization

In [ ]:
%%capture
!pip install pymupdf

In [ ]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit import IndicProcessor

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

In [ ]:
print(DEVICE)

In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text

        with tokenizer.as_target_tokenizer():
            generated_tokens = tokenizer.batch_decode(
                generated_tokens.detach().cpu().tolist(),
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

In [ ]:
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)

## Model Working

In [ ]:
en_sents = [
    "Mathematical Intelligence: This is the ability to carry out mathematical &nbsp operations.",
    "It also includes an understanding of objects and symbols used in Mathematics.",
    "Interpersonal Intelligence: This includes the ability to understand and &nbsp effectively interact with others.",
    "It also includes the ability to notice and make distinctions among moods, temperaments.",
    "Intrapersonal Intelligence: This includes the ability to understand oneself",
    "It also includes identifying what you want and don’t want, and to accept your strengths and weaknesses. When we understand and accept ourselves, we can build on our strengths.",
    "Musical Intelligence: This includes sensitivity and understanding of pitch, melody, rhythm and tone.",
    "It also includes the understanding of tones and phrases of music in the surroundings. It is an understanding of the ways to combine tones and phrases into larger musical rhythms as well as structures and includes an awareness of the emotional aspects of music.",
    "Spatial Intelligence: This includes the ability to think three-dimensionally.",
    "It includes the capacity to perceive the visual world accurately, perform transformations upon perceptions and detect similar patterns.",
    "Naturalistic Intelligence: This includes the ability to observe patterns in nature and understand natural and human-made systems.",
    "It also includes sensitivity and understanding of plants, animals and other aspects of nature.",
    "Linguistic Intelligence: This is the ability to think and use correct and appropriate words while talking and writing."
]

src_lang, tgt_lang = "eng_Latn", "hin_Deva"
hi_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
# del en_indic_tokenizer, en_indic_model

In [ ]:
hi_sents_googleAPI = [
    ["गणितीय बुद्धि: यह गणितीय संक्रियाएं करने की क्षमता है।"],
    ["इसमें गणित में प्रयुक्त वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["पारस्परिक बुद्धिमत्ता: इसमें दूसरों को समझने और उनके साथ प्रभावी ढंग से बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मनोदशाओं और स्वभावों के बीच अंतर करने और उन्हें पहचानने की क्षमता भी शामिल है।"],
    ["अंतःवैयक्तिक बुद्धि: इसमें स्वयं को समझने की क्षमता शामिल है"],
    ["इसमें यह पहचानना भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं, और अपनी ताकत और कमज़ोरियों को स्वीकार करना भी शामिल है। जब हम खुद को समझते हैं और स्वीकार करते हैं, तो हम अपनी ताकत पर काम कर सकते हैं।"],
    ["संगीत बुद्धि: इसमें सुर, राग, लय और सुर के प्रति संवेदनशीलता और समझ शामिल है।"],
    ["इसमें आसपास के वातावरण में संगीत के स्वरों और वाक्यांशों की समझ भी शामिल है। यह स्वरों और वाक्यांशों को बड़े संगीत लय और संरचनाओं में संयोजित करने के तरीकों की समझ है और इसमें संगीत के भावनात्मक पहलुओं के बारे में जागरूकता भी शामिल है।"],
    ["स्थानिक बुद्धि: इसमें त्रि-आयामी सोचने की क्षमता शामिल है।"],
    ["इसमें दृश्य जगत को सटीक रूप से देखने, धारणाओं में परिवर्तन करने और समान पैटर्न का पता लगाने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धि: इसमें प्रकृति में पैटर्न का अवलोकन करने और प्राकृतिक एवं मानव निर्मित प्रणालियों को समझने की क्षमता शामिल है।"],
    ["इसमें पौधों, जानवरों और प्रकृति के अन्य पहलुओं के प्रति संवेदनशीलता और समझ भी शामिल है।"],
    ["भाषाई बुद्धि: यह बात करते और लिखते समय सही और उपयुक्त शब्दों का प्रयोग करने और सोचने की क्षमता है।"]
]

In [ ]:
hi_sents_chatGPT = [
    ["गणितीय बुद्धिमत्ता: यह गणितीय कार्यों को करने की क्षमता है।"],
    ["इसमें गणित में उपयोग किए जाने वाले वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["अंतर-व्यक्तिगत बुद्धिमत्ता: इसमें दूसरों को समझने और उनके साथ प्रभावी ढंग से बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मनोदशा, स्वभाव और उनके बीच अंतर को पहचानने और समझने की क्षमता भी शामिल है।"],
    ["आंतरिक-व्यक्तिगत बुद्धिमत्ता: इसमें स्वयं को समझने की क्षमता शामिल है।"],
    ["इसमें यह पहचानना भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं, और अपनी ताकतों और कमजोरियों को स्वीकार करना। जब हम स्वयं को समझते और स्वीकार करते हैं, तो हम अपनी क्षमताओं पर काम कर सकते हैं।"],
    ["संगीतात्मक बुद्धिमत्ता: इसमें सुर, लय, ताल और स्वर की समझ और संवेदनशीलता शामिल है।"],
    ["इसमें आस-पास के संगीत में सुरों और वाक्यांशों की समझ भी शामिल है। यह सुरों और वाक्यांशों को बड़े संगीतात्मक तालों और संरचनाओं में संयोजित करने के तरीकों की समझ है और इसमें संगीत के भावनात्मक पहलुओं के प्रति जागरूकता भी शामिल है।"],
    ["स्थानिक बुद्धिमत्ता: इसमें त्रि-आयामी रूप से सोचने की क्षमता शामिल है।"],
    ["इसमें दृश्य दुनिया को सटीक रूप से समझने, धारणाओं पर परिवर्तन करने और समान पैटर्न को पहचानने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धिमत्ता: इसमें प्रकृति में पैटर्न का अवलोकन करने और प्राकृतिक एवं मानव-निर्मित प्रणालियों को समझने की क्षमता शामिल है।"],
    ["इसमें पौधों, जानवरों और प्रकृति के अन्य पहलुओं की संवेदनशीलता और समझ भी शामिल है।"],
    ["भाषाई बुद्धिमत्ता: यह बात करते और लिखते समय सही और उपयुक्त शब्दों का उपयोग करने और सोचने की क्षमता है।"]
]

In [ ]:
hi_sents_gemini = [
    ["गणितीय बुद्धि: यह गणितीय संक्रियाओं को करने की क्षमता है।"],
    ["इसमें गणित में प्रयुक्त वस्तुओं और प्रतीकों की समझ भी शामिल है।"],
    ["अंतरव्यक्तिगत बुद्धि: इसमें दूसरों को समझने और प्रभावी रूप से उनके साथ बातचीत करने की क्षमता शामिल है।"],
    ["इसमें मूड, स्वभाव में अंतर देखने और पहचानने की क्षमता भी शामिल है।"],
    ["अंतरात्मिक बुद्धि: इसमें स्वयं को समझने की क्षमता शामिल है।"],
    ["इसमें यह भी शामिल है कि आप क्या चाहते हैं और क्या नहीं चाहते हैं, और अपनी ताकत और कमजोरियों को स्वीकार करना। जब हम खुद को समझते हैं और स्वीकार करते हैं, तो हम अपनी ताकत पर निर्माण कर सकते हैं।"],
    ["संगीत बुद्धि: इसमें पिच, माधुर्य, लय और स्वर की संवेदनशीलता और समझ शामिल है।"],
    ["इसमें परिवेश में संगीत के स्वरों और वाक्यांशों की समझ भी शामिल है। यह स्वरों और वाक्यांशों को बड़े संगीत लय और संरचनाओं में संयोजित करने के तरीकों की समझ है और इसमें संगीत के भावनात्मक पहलुओं की जागरूकता भी शामिल है।"],
    ["स्थानिक बुद्धि: इसमें त्रि-आयामी सोचने की क्षमता शामिल है।"],
    ["इसमें दृश्य दुनिया को सटीक रूप से देखने, धारणाओं पर परिवर्तन करने और समान पैटर्न का पता लगाने की क्षमता शामिल है।"],
    ["प्राकृतिक बुद्धि: इसमें प्रकृति में पैटर्न देखने और प्राकृतिक और मानव निर्मित प्रणालियों को समझने की क्षमता शामिल है।"],
    ["इसमें पौधों, जानवरों और प्रकृति के अन्य पहलुओं की संवेदनशीलता और समझ भी शामिल है।"],
    ["भाषिक बुद्धि: यह बातचीत और लेखन के दौरान सही और उपयुक्त शब्दों का सोचने और उपयोग करने की क्षमता है।"]
]

In [ ]:
hi_sents_indicTrans2 = [[string] for string in hi_translations]

## Model Evaluation

Run the "Model Working" Section before running this section

In [ ]:
def merge_two_arrays(arr1, arr2):
    return [[arr1[i][0], arr2[i][0]] for i in range(len(arr1))]

def flatten_array(arr):
    return [sublist[0] for sublist in arr]

def merge_three_arrays(arr1, arr2, arr3):
    return [[arr1[i][0], arr2[i][0], arr3[i][0]] for i in range(len(arr1))]

In [ ]:
%%capture
!pip install evaluate

In [ ]:
# Predicttions = [Example 1, Example 2]
# references = [[Reference 1 of Example 1, Reference 2 of Example 1], [Reference 1 of Example 2, Reference 2 of Example 2]]

### BLEU Score

In [ ]:
# [Hello there] - prediction

# [[Hello there], [hello], [Hey There] ]

In [ ]:
import evaluate
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=flatten_array(hi_sents_indicTrans2), references=merge_three_arrays(hi_sents_chatGPT, hi_sents_gemini, hi_sents_googleAPI))
print(results)

In [ ]:
results = bleu.compute(predictions=flatten_array(hi_sents_googleAPI), references=merge_two_arrays(hi_sents_chatGPT, hi_sents_gemini))
print(results)

In [ ]:
results = bleu.compute(predictions=flatten_array(hi_sents_indicTrans2), references=merge_two_arrays(hi_sents_chatGPT, hi_sents_gemini))
print(results)

In [ ]:
# Translation - Referenence - Score
# GoogleAPI - ChatGPT = 0.61655
# IndicTrans - GoogleAPI = 0.6105
# IndicTrans - GoogleAPI and ChatGPT = 0.636723
# IndicTrans - ChatGPT = 0.5811
# IndicTrans - ChatGPT/GoogleAPI/GEMINI = 0.7695

### TER Score

In [ ]:
ter = evaluate.load("ter")
results = ter.compute(predictions=flatten_array(hi_sents_indicTrans2), references=merge_three_arrays(hi_sents_chatGPT, hi_sents_gemini, hi_sents_googleAPI))
print(results)

### Meteor Score

In [ ]:
meteor = evaluate.load("meteor")
results = meteor.compute(predictions=flatten_array(hi_sents_indicTrans2), references=merge_three_arrays(hi_sents_chatGPT, hi_sents_gemini, hi_sents_googleAPI))
print(results)

# App

## Imports


In [ ]:
import pymupdf as fitz
import re

## Convert PDF

In [ ]:
def checkText(text: str):
  text = text.strip()
  if(text == '\t' or text == '•' or text == ' ' or text == ""):
    return False
  if all(char == '.' for char in text):
    return False
  if re.search(r'\d+[\t\n]', text):
    return False
  try:
    float(text)
    return False
  except:
    pass
  if text.isspace():
      return False
  return True

In [ ]:
def convertPDF(name, tgt_lang="hin_Deva"):
  doc = fitz.open(f"{name}.pdf")
  for i in range(len(doc)):
    print(i+1)
    page = doc[i]
    content_block = page.get_text("dict")["blocks"]
    text_blocks = []
    bboxes = []
    font_sizes = []
    colors = []
    for block in content_block:
      if block["type"] == 0:
        for line in block["lines"]:
          for span in line["spans"]:
            if(checkText(span["text"])):
              text_blocks.append(span["text"])
              bboxes.append(span["bbox"])
              font_sizes.append(span["size"])
              color = fitz.sRGB_to_rgb(span["color"])
              if(color == (255,255,255)):
                color = (0,0,0)
              colors.append(color)

    src_lang = "eng_Latn"
    textsTranslated = batch_translate(text_blocks, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

    if(len(bboxes) != len(textsTranslated)):
      print("Error")
      pass

    for i in range(len(bboxes)):
      css = "*{color:" + f"rgb{colors[i]};font-size: {font_sizes[i]}px;" + "}"
      page.draw_rect(bboxes[i], color=(1, 1, 1), fill=(1, 1, 1))
      page.insert_htmlbox(bboxes[i],textsTranslated[i], css=css)

  doc.save(f"{name}_Converted.pdf")
  pass

## Convert TXT

In [ ]:
def convertTxt(name):
  with open(f'{name}.txt', 'r') as file:
    lines = file.readlines()

  src_lang, tgt_lang = "eng_Latn", "hin_Deva"
  hi_translations = batch_translate(lines, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

  fullContent = ""
  for line in hi_translations:
    fullContent += line + "\n"

  with open('output.txt', 'w') as outPutTxt:
    outPutTxt.write(fullContent)
  pass

## Convert DOCX

In [ ]:
def convertDocx(name):
  pass

## ConvertFile

In [ ]:
def convertFile(name, type: str, tgt_lang="hin_Deva"):
  if(type == "pdf"):
    convertPDF(name, tgt_lang)
  elif(type == "txt"):
    convertTxt(name)
  elif(type == "docx"):
    convertDocx(name)
  else:
    print("Invalid Type")
  pass

In [ ]:
[nameOfFile , typeOfFile] = "sample1.pdf".split(".")

convertFile(nameOfFile, typeOfFile, "tam_Taml")

In [ ]:
# Marathi = "mar_Deva"
# Gujarathi = "guj_Gujr"
# Hindi = "hin_Deva"
# Tamil = "tam_Taml"
# Telugu = "tel_Telu"